如果你既需要**定时自动醒**，又需要**能被随时提前唤醒**，Qt 的 `QWaitCondition` 完美支持这种需求。你只需要在调用 `wait` 时传入一个**超时时间（毫秒）**即可。

以下是修改后的代码，实现了“每 5000 毫秒自动醒一次执行任务，但也可以随时被提前唤醒”：

```python
from PySide6.QtCore import QThread, QMutex, QWaitCondition, QMutexLocker

class WorkerThread(QThread):
    def __init__(self):
        super().__init__()
        self.mutex = QMutex()
        self.wait_condition = QWaitCondition()
        self._is_interrupted = False

    def run(self):
        while True:
            with QMutexLocker(self.mutex):
                # 1. 优先检查是否被请求彻底退出
                if self._is_interrupted:
                    break
                
                # 2. 带超时的等待（例如 5000 毫秒 = 5 秒）
                # 如果 5 秒内没人唤醒，它会自动醒来返回 False
                # 如果期间被 wakeOne/wakeAll 提前唤醒，它会立刻醒来返回 True
                # 注意：这里使用了 while 循环来应对系统可能产生的“虚假唤醒”
                while not self._is_interrupted:
                    # wait 返回 False 代表超时（到时间自动醒），True 代表被提前唤醒
                    condition_met = self.wait_condition.wait(self.mutex, 5000) 
                    if not condition_met:
                        print("5秒时间到，自动醒来执行任务...")
                    else:
                        print("被提前唤醒，执行任务...")
                    break # 无论是超时还是被唤醒，都跳出循环去执行 do_work[[source_group_web_3]]

            # 3. 在这里执行你的耗时任务（此时锁已经释放）
            self.do_work()

    def interrupt(self):
        """外部调用，彻底中断并退出线程"""
        with QMutexLocker(self.mutex):
            self._is_interrupted = True
            self.wait_condition.wakeAll()

    def wake_up(self):
        """外部调用，提前唤醒休眠的线程"""
        with QMutexLocker(self.mutex):
            self.wait_condition.wakeOne()

    def do_work(self):
        print("正在执行业务逻辑...")
```

### 💡 核心改动点解析：
1. **`wait(self.mutex, 5000)`**：增加了第二个参数（超时毫秒数）。这样线程就有了“保底醒来”的时间。
2. **返回值判断**：`wait` 方法会返回一个布尔值。
   * **`False`**：表示等待超时（时间到了，自己醒的）。
   * **`True`**：表示在超时之前，被其他线程通过 `wakeOne()` 或 `wakeAll()` 提前唤醒了。
3. **`while` 循环**：这是多线程编程中的一个最佳实践，用于防范“虚假唤醒”（即系统偶尔可能会在没有收到任何信号的情况下意外唤醒线程）。

通过这种方式，你的线程就同时具备了**定时休眠**和**随时中断唤醒**的双重能力。